# Les files — Exercice 6 : une pile avec deux files

**Notions travaillées :** implémenter une structure (pile, LIFO) **à l'aide d'une autre** (file, FIFO), invariant d'une classe, coût des opérations.

> **Rappel :** une **pile** fonctionne en mode **LIFO** (*Last In, First Out*) : on dépile le **dernier** élément empilé. Méthodes : `empile(el)`, `depile()`, `est_vide()`, `taille()`.

**La difficulté :** une file ne donne accès qu'à son **premier** élément, alors qu'une pile doit renvoyer le **dernier**. Il va falloir faire défiler tout le monde pour atteindre le dernier.

## Mise en place

On importe la classe `File` écrite à l'exercice 1 (le fichier `file.py` doit être **dans le même dossier** que ce notebook).

L'énoncé interdit d'utiliser le type `list` : on ne manipule les files **qu'avec leurs méthodes** `enfile`, `defile`, `taille`, `est_vide`. La petite fonction `file_de` sert uniquement à fabriquer rapidement des files pour les tests.

On affiche une file sous la forme `tête < ... < queue` : on **défile à gauche** et on **enfile à droite**.

In [1]:
from file import File   # la classe File de l'exercice 1 (fichier file.py placé à côté du notebook)


def file_de(*elements):
    """Outil pour les tests : construit une file contenant les éléments donnés (le premier en tête)."""
    f = File()
    for el in elements:
        f.enfile(el)
    return f


print(file_de("a", "b", "c"))

tête < 'a' < 'b' < 'c' < queue


## Les indications, une par une

- **« Une des deux files est toujours vide »** : c'est un **invariant**, une propriété vraie après chaque appel de méthode. Tous les éléments sont dans **une seule** des deux files, dans l'ordre où ils ont été empilés.
- **« La pile est vide lorsque les deux files sont vides »** → `est_vide`.
- **`taille`** : le nombre d'éléments est la somme des tailles des deux files (l'une vaut 0).
- **« Pour empiler, on enfile dans la file non vide »** (si les deux sont vides, peu importe : on prend la première).
- **« Pour dépiler, on transfère les éléments de la file non vide vers l'autre, sauf le dernier, qui est renvoyé »** : après le transfert, c'est l'**autre** file qui contient les éléments, et l'invariant est respecté.

On écrit une petite méthode `_pleine()` qui renvoie le couple *(file non vide, file vide)*, pour ne pas répéter ce test dans chaque méthode. (Le `_` au début du nom signale une méthode **interne**, qui ne fait pas partie de l'API.)

**Exemple :** on empile 1, 2, 3, puis on dépile.

| Étape | `f1` | `f2` |
|:---|:---|:---|
| après les empilements | `1 2 3` | |
| transfert de 1 et 2 | `3` | `1 2` |
| on défile et renvoie 3 | | `1 2` |

In [2]:
class Pile:

    def __init__(self):
        self.f1 = File()
        self.f2 = File()

    def _pleine(self):
        """Renvoie (file non vide, file vide). Si les deux sont vides, renvoie (f1, f2)."""
        if self.f2.est_vide():
            return self.f1, self.f2
        return self.f2, self.f1

    def est_vide(self):
        return self.f1.est_vide() and self.f2.est_vide()

    def taille(self):
        return self.f1.taille() + self.f2.taille()

    def empile(self, el):
        pleine, vide = self._pleine()
        pleine.enfile(el)

    def depile(self):
        if self.est_vide():
            raise IndexError("depile() sur une pile vide")
        pleine, vide = self._pleine()
        while pleine.taille() > 1:            # on transfère tout sauf le dernier
            vide.enfile(pleine.defile())
        return pleine.defile()                # le dernier empilé

    def __repr__(self):
        pleine, vide = self._pleine()
        return "Pile (sommet à droite) : " + repr(pleine)

## Tests

On vérifie le **LIFO**, l'alternance d'empilements et de dépilements, l'**invariant** et le cas d'erreur.

In [3]:
p = Pile()
assert p.est_vide() and p.taille() == 0
for x in [1, 2, 3]:
    p.empile(x)
print(p)
assert p.taille() == 3
assert p.depile() == 3           # le dernier empilé sort en premier
p.empile(4)
print(p)
assert p.depile() == 4
assert p.depile() == 2
assert p.depile() == 1
assert p.est_vide()

# invariant : après chaque opération, au moins une des deux files est vide
p = Pile()
for x in range(10):
    p.empile(x)
    assert p.f1.est_vide() or p.f2.est_vide()
for x in range(9, -1, -1):
    assert p.depile() == x
    assert p.f1.est_vide() or p.f2.est_vide()

try:
    p.depile()
except IndexError as e:
    print("Erreur bien déclenchée :", e)
print("Tous les tests passent ✔")

Pile (sommet à droite) : tête < 1 < 2 < 3 < queue
Pile (sommet à droite) : tête < 1 < 2 < 4 < queue
Erreur bien déclenchée : depile() sur une pile vide
Tous les tests passent ✔


💡 **Coût des opérations :** `empile` fait un seul `enfile` (coût constant), mais `depile` déplace **tous** les éléments sauf un : son coût est proportionnel à la taille de la pile. C'est le prix à payer pour simuler une pile avec des files. On pourrait faire l'inverse (empilement coûteux, dépilement rapide).

⚠️ **Erreurs fréquentes**
- Transférer **tous** les éléments, y compris le dernier : on ne sait plus lequel renvoyer.
- Toujours enfiler dans `f1` : après un premier `depile`, les éléments sont dans `f2`, et l'invariant est cassé.